# Gaussian Blur Interactive Demo

This notebook showcases a real use case for the numerical solution of heat equation: Gaussian blur, which is an image smoothing technique that works by replacing each pixel's value with the average of the pixels around it. Mathematically, this is identical to running the heat equation for some time steps with the image as an initial condition. The diffusivity constant and number of time steps determines how much the image is blurred. This relationship is exactly as follows. A blur of radius $\sigma$ corresponds to running the heat equation up to time $t=\frac{\sigma^2}{2\alpha}$.

This effect is used in image processing, often for noise reduction and background blur effects. It is also used frequently in computer vision to remove noise and prevent unwanted edges from being detected.

In [1]:
# Imports
import numpy as np
import matplotlib.pyplot as plt
from finite_differences import Grid, Field
from finite_differences.bc import Dirichlet
from finite_differences.operators import laplacian, laplacian_5pt
from finite_differences.time_integrators import explicit_euler, crank_nicolson
from finite_differences.solvers import LinearSolver
from finite_differences.utils import quick_plot_2d
from PIL import Image

As an example, we will use this image of my dog and apply Gaussian blur.

![My dog](images/dog.JPG)

In [ ]:
# Load image in
img = np.transpose(np.array(Image.open("images/dog.JPG"), dtype=float))
imgshape = img.shape # (3, 128, 228)

nx = imgshape[1]
ny = imgshape[2]
dt = 5e-5
blur_radius = 2
alpha = 2
steps = 100


(128, 228) (3, 128, 228)


Then we use the following helper functions to set up the finite difference formula for the heat equation,.


In [ ]:
def rhs_heat(field, alpha):
    # apply BCs must be done before calling
    lap = laplacian(field)
    return alpha * lap

def run_explicit(u0, nx=64, ny=64, alpha=1e-3, dt=1e-4, steps=100):
    grid = Grid(nx, ny, lx=10.0, ly=10.0)
    f = Field(grid, ng=1)
    X, Y = grid.mesh()
    f.set_interior(u0)
    bc = Dirichlet(0.0)
    interior_history = []
    for n in range(steps):
        bc.apply(f)
        interior_history.append(f.interior.copy())
        u_interior = f.interior
        u_new = explicit_euler(u_interior, lambda u: rhs_heat(f, alpha), dt)
        # write back
        f.set_interior(u_new)
    bc.apply(f)
    return np.asarray(interior_history)

We execute the finite difference method on each pixel value individually, then combine the resulting fields to recreate the blurred image.

In [ ]:
blurred_img = np.zeros(np.append(steps, imgshape))

for i in range(3):
    blurred_img[:,i,:,:] = run_explicit(img[i,:,:], nx, ny, alpha, dt, steps)

In [50]:
print(blurred_img.shape)

final_img = np.transpose(blurred_img[50,:,:,:])
final = Image.fromarray(final_img.astype(np.uint8))
final.save('images/blurreddog.JPG')

(20000, 3, 128, 228)


Original image and blurred image side-by-side:

![Regular dog](images/dog.JPG)
![Blurred dog](images/blurreddog.JPG)